# Investigating Factors Contributing to High Star Ratings on Yelp

- **Author**: Ervin Pangilinan
- COSC 526: Data Mining & Analytics - Spring 2025


## Data Preprocessing

### Import Libraries for Pre-Processing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from sklearn.preprocessing import StandardScaler

# Suppress warnings output
import warnings
warnings.filterwarnings("ignore")

Once we load the raw data, we will need to extract the relevant features for our analysis.

In [ ]:
business_df = pd.read_json('yelp_academic_dataset_business.json', lines=True)

# Drop unnecessary columns
business_df = business_df.drop(columns=['is_open', 'business_id'])
business_df.head(10)

In [ ]:
business_df.info()

After loading the data, we see that we will need to filter the data to only include restaurants that are in the data. Next, we'll have to do additional preprocessing to clean the data and only include restaurants.

In [ ]:
# Filter for businesses with the restaurant attribute
business_df = business_df[business_df['categories'].str.contains('Restaurant', na=False)]
business_df.head(10)

### Dealing with Missing Values

Next, let's check for missing values in the data. We will need to drop any rows that have missing values in the features we are interested in. We will also need to convert the data types of some of the columns to make them easier to work with.

In [ ]:
# Replace cells with only blank spaces in any column with NaN and check for missing data
business_df = business_df.applymap(lambda x: np.nan if isinstance(x, str) and not x.strip() else x)
missing_data = business_df[business_df.isnull().any(axis=1)]
missing_data.head(10)

In [ ]:
missing_data_count = business_df.isnull().sum()
print(f"Missing data count:\n{missing_data_count}")

# Total number of rows with missing data
total_missing_data = missing_data.shape[0]
print(f"\nTotal number of rows with missing data: {total_missing_data}")

We see that there out of the 52286 samples in our dataset, 7800 of them have missing values. Further investigation is needed to see which features have missing values and how we can handle them.

In [ ]:
# Calculate the percentage of missing values in each column
missing_percentage = business_df.isnull().mean()
missing_percentage

##### Missing Attributes
Since samples with missing attributes only make up about 1% of this reduced dataset, we can drop them.

In [ ]:
# Drop rows with missing attributes
business_df = business_df.dropna(subset=['attributes'])

# Check the number of rows after dropping missing values
print(f"Number of rows after dropping missing values: {business_df.shape[0]}")

##### Missing Hours and Transforming the Feature

In this dataset, the hours are listed as a dictionary with the days of the week as keys and the hours of operation as values. We will need to convert this to a more usable format. We will transform this feature into hours open during the week and hours open on weekends. 

In [ ]:
# Transform the hours column into 2 columns: number of hours open during weekdays and weekends
def transform_hours(hours):
    if not isinstance(hours, dict):
        return pd.Series([np.nan, np.nan])
    weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    weekend = ['Saturday', 'Sunday']
    weekday_hours = sum([len(v) for k, v in hours.items() if k in weekdays])
    weekend_hours = sum([len(v) for k, v in hours.items() if k in weekend])
    return pd.Series([weekday_hours, weekend_hours])

if 'hours' in business_df.columns:
    business_df[['weekday_hours', 'weekend_hours']] = business_df['hours'].apply(transform_hours)
    business_df = business_df.drop(columns=['hours'])

# Check the first few rows of the transformed DataFrame
business_df.head(10)

We don't want to drop the rows with missing values in the hours column because it made up 14% of our dataset. Instead, we will use the median value to fill in the missing values since samples that had missing hours values will in turn have missing values in the hours open during the week and hours open on weekends.

In [ ]:
# Fill in missing values in the weekday_hours and weekend_hours columns with with the median value of the column
weekday_hours_mode = business_df['weekday_hours'].median()
weekend_hours_mode = business_df['weekend_hours'].median()
business_df['weekday_hours'] = business_df['weekday_hours'].fillna(weekday_hours_mode)
business_df['weekend_hours'] = business_df['weekend_hours'].fillna(weekend_hours_mode)

# Check the first few rows of the DataFrame after filling missing values
business_df.head(10)

#### Flattening Raw Data for Features

In [ ]:
# Flatten the DataFrame completely
def flatten_column(df, col):
    # Expand the first-level attributes dictionary
    df = df.copy()
    attributes_expanded = pd.json_normalize(df[col])

    # Merge carefully by resetting indices
    df = df.reset_index(drop=True)
    attributes_expanded = attributes_expanded.reset_index(drop=True)
    df = pd.concat([df, attributes_expanded], axis=1)
    
    # Drop the original attributes column
    df = df.drop(columns=[col])

    return df

business_df = flatten_column(business_df, 'attributes')
business_df.head(10)

Let's check for missing data again.

In [ ]:
# Check percentage of missing values in the flattened DataFrame
missing_percentage = business_df.isnull().mean()
print(f"Missing percentage in flattened DataFrame:\n{missing_percentage}")

We'll drop the columns where the percentage of missing values is greater than 50%.

In [ ]:
# Drop columns with more than 50% missing values
missing_percentage = business_df.isnull().mean()
columns_to_drop = missing_percentage[missing_percentage >= 0.5].index
business_df = business_df.drop(columns=columns_to_drop)

# Check the number of rows and columns after dropping columns
print(f"Number of rows and columns after dropping columns: {business_df.shape}")

missing_percentage = business_df.isnull().mean()
print(f"Missing percentage in flattened DataFrame:\n{missing_percentage}")

business_df.head(10)

#### Cleaning Usable Features
We will also need to convert the data types of some of the columns to make them easier to work with.

In [ ]:
bool_columns = [
    'RestaurantsDelivery', 'OutdoorSeating', 'BusinessAcceptsCreditCards',
    'RestaurantsReservations', 'Caters', 'BikeParking', 'RestaurantsTakeOut',
    'RestaurantsGoodForGroups', 'GoodForKids', 'HasTV'
]

for col in bool_columns:
    # Perform one-hot encoding for boolean columns
    business_df[col] = business_df[col].replace({None: False})
    business_df[col] = business_df[col].astype(bool)
    business_df[col] = business_df[col].replace({True: 1, False: 0})

missing_percentage = business_df.isnull().mean()
print(f"Missing percentage in flattened DataFrame:\n{missing_percentage}")

business_df.head(10)

In [ ]:
# Clean the 'RestaurantsPriceRange2' column
business_df['RestaurantsPriceRange2'] = pd.to_numeric(business_df['RestaurantsPriceRange2'], errors='coerce')
business_df['RestaurantsPriceRange2'] = business_df['RestaurantsPriceRange2'].fillna(business_df['RestaurantsPriceRange2'].mode()[0])

# Drop BusinessParking and GoodForMeal columns
business_df = business_df.drop(columns=['BusinessParking', 'GoodForMeal', 'Ambience'])
business_df.head(10)

#### Dealing with Missing Values from Remaining Features
There are still some missing values in the remaining features. We will need to fill in the missing values for the remaining features.

In [ ]:
# Check percentage of missing values
missing_percentage = business_df.isnull().mean()

# Print the column names with missing values percentage greater than 1%
missing_columns = missing_percentage[missing_percentage >= 0.01].index
missing_percentage = business_df[missing_columns].isnull().mean()

print(f"Features with 1% or more for missing data:\n{missing_percentage}")

Since these features are categorical, we will need to fill in the missing values with the mode of the column. We will also need to convert the data types of some of the columns to make them easier to work with.

In [ ]:
columns_to_fill = ['WiFi', 'Alcohol', 'RestaurantsAttire', 'NoiseLevel']

# Fill missing values with the mode of the column
for col in columns_to_fill:
    business_df[col] = business_df[col].fillna(business_df[col].mode()[0])
    
# Perform one-hot encoding for categorical columns
business_df = pd.get_dummies(business_df, columns=columns_to_fill, drop_first=True)

# Check if column is boolean and change to 0/1 if necessary
for col in business_df.columns:
    if business_df[col].dtype == 'bool':
        business_df[col] = business_df[col].astype(int)
  
# Check the first few rows of the DataFrame after filling missing values
business_df.head(10)

### Extracting Features from the Categories Column
We will need to extract the features from the categories column. We will need to convert the categories into a one-hot encoded format so that we can use them in our analysis. We will also need to drop the original categories column.

See the following for more information on categories provided by Yelp: https://blog.yelp.com/businesses/yelp_category_list/

In [ ]:
business_df['categories_list'] = business_df['categories'].fillna('').apply(lambda x: [i.strip() for i in x.split(',')])

# Drop the original categories column
business_df = business_df.drop(columns=['categories'])
business_df.head(10)

In [ ]:
# Finding the most common categories
categories = business_df['categories_list'].explode()
categories = categories.str.strip()
categories = categories[categories != '']
categories = categories.value_counts()
categories = categories[categories > 100]
categories = categories.reset_index()
categories.columns = ['category', 'count']
categories = categories.sort_values(by='count', ascending=False)

# Remove the 'Restaurants' category and 'Food' category
categories = categories[~categories['category'].isin(['Restaurants', 'Food', 'Event Planning & Services', 'Caterers'])]

# Remove the 'Restaurants' and 'Food' categories from the DataFrame
business_df['categories_list'] = business_df['categories_list'].apply(lambda x: [i for i in x if i not in ['Restaurants', 'Food', 'Event Planning & Services', 'Caterers']])

# Export data to CSV
business_df.to_csv('yelp_business_with_categories.csv', index=False)

# Plot the top 25 categories
plt.figure(figsize=(12, 6))
sns.barplot(x='count', y='category', data=categories.head(25))
plt.title('Top 25 Categories for Restaurants in the Yelp Dataset')
plt.xlabel('Count')
plt.ylabel('Category')
plt.show()

Now that we know what the most common categories are, we can perform one-hot encoding on the categories column.

In [ ]:
# One-hot encoding for the top 25 categories
top_25_categories = categories['category'].head(25).tolist()
for category in top_25_categories:
    business_df[category] = business_df['categories_list'].apply(lambda x: 1 if category in x else 0)
    
# Drop the categories_list column
business_df = business_df.drop(columns=['categories_list'])
business_df.head(10)

### Finalizing the Dataset

We'll be using regression and clustering later on, so we will modify this DataFrame to be more usable for those tasks. For 
regression, we will remove and latitude from the DataFrame. However, we will keep them for clustering. We will also remove
unnecesary columns such as the name and address of the restaurant.

In [ ]:
columns_to_drop = ['name', 'address', 'city', 'state', 'postal_code']

# Extract the above columns to a new DataFrame
raw_data = business_df.copy()
raw_data.to_csv('raw_data.csv', index=False)
business_info_df = business_df[columns_to_drop].copy()

# Drop the columns from the original DataFrame
business_df = business_df.drop(columns=columns_to_drop)

# Standardize the numerical columns, except for latitude, longitude, and stars
scaler = StandardScaler()
numerical_columns = business_df.select_dtypes(include=[np.number]).columns.tolist()
columns_to_scale = [col for col in numerical_columns if col not in ['longitude', 'latitude', 'stars']]
business_df[columns_to_scale] = scaler.fit_transform(business_df[columns_to_scale])

# Making separate datasets for regression and clustering
business_clustering_df = business_df.copy()
business_regression_df = business_df.copy().drop(columns=['longitude', 'latitude'])

# Export the DataFrames to CSV files
business_clustering_df.to_csv('business_clustering_df.csv', index=False)
business_regression_df.to_csv('business_regression_df.csv', index=False)
business_info_df.to_csv('business_info_df.csv', index=False)

## Data Exploration

First, we need to see the locations of the different restaurants in the dataset.



### Data Overview and Check for Outliers

In [ ]:
business_df = pd.read_csv('raw_data.csv')
business_df.describe()

This dataset after pre-processing has 68 features. We see one outlier, which is that a restaurant has 7568 reviews.

In [ ]:
# Find the restaurant that has 7568 reviews using business_info_df
restaurant_with_7568_reviews = business_df[business_df['review_count'] == 7568]
restaurant_with_7568_reviews[['name', 'address', 'city', 'state', 'review_count', 'stars']]

Further inspection shows that this restaurant has been around since 1910. It's useful real-world data worth keeping even though it is an outlier. We will keep this restaurant in the dataset.

### Visualizing the Locations of Restaurants

In [ ]:
# Create a 'geometry' column
business_df['geometry'] = business_df.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)

# Convert regular DataFrame to GeoDataFrame
gdf = gpd.GeoDataFrame(business_df, geometry='geometry')

# Set the coordinate reference system (CRS) - EPSG:4326 is WGS84 standard (lat/lon)
gdf.set_crs(epsg=4326, inplace=True)

# Setup plot
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.PlateCarree()})

# Plot Yelp points
gdf.plot(ax=ax, markersize=5, alpha=0.7, color='red')

# Add map features
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':', edgecolor='black', linewidth=1)
ax.add_feature(cfeature.STATES, linestyle='-', edgecolor='gray', linewidth=0.5)        
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)                                      
ax.set_title('Yelp Restaurants Mapped')
plt.show()

### Exploring the Distribution of Star Ratings

We would like to see the distribution of star ratings in the dataset to see if the dataset is unbalanced.

In [ ]:
# Plot the distribution of star ratings as a pie chart
star_distribution = business_df['stars'].value_counts(normalize=True)
plt.figure(figsize=(7, 7))
plt.pie(star_distribution, labels=star_distribution.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribution of Star Ratings for Yelp Restaurants')
plt.axis('equal')
plt.show()

We see that the distribution is heavily skewed. Almost half of the restaurants in the dataset have a star rating of either 3.5 or 4.0. This could pose a problem for our analysis since we will be using regression later on.

### Correlation of Features to Star Ratings

Our regression model will be used to predict the star ratings based on the features we have. We will need to see how the features are correlated with the star ratings. We will also need to see how the features are correlated with each other.

We would like to investigate the following: **What factors contribute to high star ratings on Yelp?**

In [ ]:
# Select numeric columns
numerical_cols = business_df.select_dtypes(include=['number']).columns

# Correlation matrix
corr_matrix = business_df[numerical_cols].corr()

# Drop 'stars' itself (perfect correlation)
stars_corr = corr_matrix['stars'].sort_values(ascending=False)
stars_corr = stars_corr.drop('stars')

# Split into positive and negative parts
positive_corr = stars_corr[stars_corr > 0]
negative_corr = stars_corr[stars_corr < 0]
top_positive_corr = stars_corr[stars_corr > 0].sort_values(ascending=False).head(10)
top_negative_corr = stars_corr[stars_corr < 0].sort_values().head(10)

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# Plot positive correlations
top_positive_corr.sort_values().plot(kind='barh', color='seagreen', ax=axes[0])
axes[0].set_title('Top 10 Positive Correlations with Star Ratings')
axes[0].set_xlabel('Correlation Coefficient')
axes[0].grid(axis='x', linestyle='--', alpha=0.7)

# Plot negative correlations
top_negative_corr.sort_values(ascending=False).plot(kind='barh', color='salmon', ax=axes[1])
axes[1].set_title('Top 10 Negative Correlations with Star Ratings')
axes[1].set_xlabel('Correlation Coefficient')
axes[1].grid(axis='x', linestyle='--', alpha=0.7)

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

From this plot, we can see that the star ratings are positively correlated with:

- restaurant price range
- review counts
- if the restaurant is a cafe

Even though these three features were the most positively correlated, the positive correlation is weak, compared to the top features for negative correlations.

We can also see that the star ratings are negatively correlated with:
- if the restaurant serves fast food
- if the restaurant serves burgers
- if the restaurant serves chicken wings

Specific food categories were surprising to see. Upon first glance, this could be due to the fact that these food categories are more likely to be fast food restaurants. However, seeing the fast food category as the most negatively correlated is realistic. In general, the food quality would be lower than traditional restaurants. Additionally, reviewers may tend to rate these chains harshly such as when an order is wrong or takes too long at the drive-thru. 

## Regression Analysis

We would like to answer the following question through regression models: **Can we predict star ratings using various machine learning models?**

For our regression analysis, we will be using tree-based models because of the one-hot encoding that was implemented during pre-processing. We will be using the following models:
- Random Forest
- Histogram-based Gradient Boosting

### Importing Libraries for Regression Models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GridSearchCV
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def evaluate(y_true, y_pred):
    print(f'MSE:  {mean_squared_error(y_true, y_pred):.4f}')
    print(f'RMSE: {root_mean_squared_error(y_true, y_pred):.4f}')
    print(f'MAE:  {mean_absolute_error(y_true, y_pred):.4f}')
    print(f'R^2:  {r2_score(y_true, y_pred):.4f}')

### Splitting the Dataset

In [ ]:
business_regression_df = pd.read_csv('business_regression_df.csv')

# Split the data into training and testing sets
X = business_regression_df.drop(columns=['stars'])
y = business_regression_df['stars']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

# Check the distribution of the target variable in the training set
print(f"Training set target variable distribution:\n{y_train.value_counts(normalize=True)}")

# Check the distribution of the target variable in the testing set
print(f"Testing set target variable distribution:\n{y_test.value_counts(normalize=True)}")

### Random Forest Regressor

We'll first use GridSearch to figure out the optimal hyperparameters to use for our Random Forest Regressor.

In [ ]:
param_grid = {
    'n_estimators': [100, 300, 500],          
    'max_depth': [10, 20, 30, None],            
    'min_samples_split': [2, 5, 10],            
}

rf = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Score (MSE):", -grid_search.best_score_)

GridSearch results tell us that the optimal configuration would have:
- max_depth: 20
- min_samples_split: 10
- n_estimators: 500

In [ ]:
rf = RandomForestRegressor(n_estimators=500, max_depth=20, min_samples_split=10, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Evaluate the model
print("Random Forest Regressor Scores")
evaluate(y_test, y_pred_rf)

# Feature importance
rf_importances = rf.feature_importances_
rf_feature_names = X_train.columns
rf_indices = np.argsort(rf_importances)[::-1]

plt.figure(figsize=(12, 6))
plt.title("Feature Importances with Random Forest Regressor")
plt.bar(range(X_train.shape[1]), rf_importances[rf_indices], align="center")
plt.xticks(range(X_train.shape[1]), rf_feature_names[rf_indices], rotation=90)
plt.xlim([-1, X_train.shape[1]])
plt.ylabel("Relative Importance")
plt.xlabel("Features")
plt.tight_layout()
plt.show()

### Histogram-based Gradient Boosting Regressor

In [ ]:
hgb_model = HistGradientBoostingRegressor(random_state=42)

param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_iter': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'l2_regularization': [0, 0.1, 1.0],
    'max_leaf_nodes': [31, 50, 100]
}

grid_search_hgb = GridSearchCV(
    estimator=hgb_model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
)

grid_search_hgb.fit(X_train, y_train)

print("Best Parameters:", grid_search_hgb.best_params_)
print("Best Score (MSE):", -grid_search_hgb.best_score_)

We see that the optimal hyperparameter values are:
- l2_regularization: 1.0
- learning_rate: 0.05
- max_depth: 7
- max_iter: 300
- max_leaf_nodes: 50

In [ ]:
# Fit the Histogram-based Gradient Boosting Regressor with the best parameters
hgb_model = HistGradientBoostingRegressor(
    l2_regularization=1.0,
    learning_rate=0.05,
    max_depth=7,
    max_iter=300,
    max_leaf_nodes=50
)

hgb_model.fit(X_train, y_train)
y_pred_hgb = hgb_model.predict(X_test)

# Evaluate the model
print("Histogram-based Gradient Boosting Regressor Scores")
evaluate(y_test, y_pred_hgb)

result = permutation_importance(hgb_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
hgb_importances = result.importances_mean
hgb_feature_names = X_train.columns
hgb_indices = np.argsort(hgb_importances)[::-1]

plt.figure(figsize=(12, 6))
plt.title("Feature Importances with Histogram-based Gradient Boosting Regressor")
plt.bar(range(X_train.shape[1]), hgb_importances[hgb_indices], align="center")
plt.xticks(range(X_train.shape[1]), hgb_feature_names[hgb_indices], rotation=90)
plt.xlim([-1, X_train.shape[1]])
plt.ylim(0, None)
plt.ylabel("Relative Importance")
plt.xlabel("Features")
plt.tight_layout()
plt.show()

### Feature Selection for Model Improvement

Many of the features have little importance. Let's try to improve our model's performance by getting rid of features with zero or almost-zero feature importances. 

In [ ]:
# Feature Selection based on average importances from both models
avg_importances = (rf_importances + hgb_importances) / 2
avg_feature_names = X_train.columns

importance_df = pd.DataFrame({
    'Feature': avg_feature_names,
    'Importance': avg_importances
})

selected_features = importance_df[importance_df['Importance'] > 0.002]['Feature'].tolist()
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Print out the selected features, 2 features per row
print("Selected Features:")
for i in range(0, len(selected_features), 2):
    print(f"{selected_features[i]:<30} {selected_features[i+1] if i+1 < len(selected_features) else ''}")

print(f"\nNumber of selected features: {len(selected_features)}")

In [ ]:
# Refit the Random Forest Regressor with selected features
rf_selected = RandomForestRegressor(n_estimators=500, max_depth=20, min_samples_split=10, random_state=42)
rf_selected.fit(X_train_selected, y_train)
y_pred_rf_selected = rf_selected.predict(X_test_selected)

# Refit the Histogram-based Gradient Boosting Regressor with selected features
hgb_model_selected = HistGradientBoostingRegressor(
    l2_regularization=1.0,
    learning_rate=0.05,
    max_depth=7,
    max_iter=300,
    max_leaf_nodes=50
)
hgb_model_selected.fit(X_train_selected, y_train)
y_pred_hgb_selected = hgb_model_selected.predict(X_test_selected)

# Evaluate the models with selected features
print("Random Forest Regressor with Selected Features Scores")
evaluate(y_test, y_pred_rf_selected)
print("\nHistogram-based Gradient Boosting Regressor with Selected Features Scores")
evaluate(y_test, y_pred_hgb_selected)

# Plot the feature importances of the selected features for both models
plt.figure(figsize=(12, 6))
plt.title("Feature Importances with Selected Features")

# Map selected feature names to their indices
selected_indices = [list(rf_feature_names).index(feature) for feature in selected_features]

# Plot the feature importances of the selected features for both models
plt.bar(range(len(selected_features)), rf_importances[selected_indices], align="center", label='Random Forest')
plt.bar(range(len(selected_features)), hgb_importances[selected_indices], align="center", label='Histogram-based Gradient Boosting', alpha=0.7)
plt.xticks(range(len(selected_features)), selected_features, rotation=90)
plt.xlim([-1, len(selected_features)])
plt.ylim(0, None)
plt.ylabel("Relative Importance")
plt.xlabel("Features")
plt.legend()
plt.tight_layout()
plt.show()

## Clustering Analysis

With clustering, we would like to answer the following: **What are the geographical hotspots for certain restaurant types?**

To answer this question, we need to reduce the dataset so that we can focus on a particular city. By reducing the dataset, we focus 
on hotspots for a particular city and observe if the trends are consistent across another city. For this clustering problem, we
will cluster the restaurants located in Nashville, TN.

#### Importing Libraries for Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import ast
from collections import Counter
import pandas as pd

#### Filtering the Dataset for Nashville

In [ ]:
# Filter the restaurants for only restaurants in Nashville
business_info_df = pd.read_csv('business_info_df.csv')
nashville_restaurants = business_info_df[business_info_df['city'] == 'Nashville']
nashville_restaurants_row_numbers = nashville_restaurants.index.tolist()
nashville_restaurants.head(10)

Now that we have the row numbers for the Nashville restaurants, we can use those row numbers to extract the appropriate rows of 
features in the clustering CSV file.

In [ ]:
# Use the row numbers to extract the corresponding rows from the clustering dataset
business_clustering_df = pd.read_csv('business_clustering_df.csv')
nashville_restaurants_clustering = business_clustering_df.iloc[nashville_restaurants_row_numbers]
nashville_restaurants_clustering.head(10)

In [ ]:
nashville_restaurants_clustering.info()

#### Clustering the Reduced Dataset

We will use K-Means to cluster the dataset. First, we'll determine the appropriate k-value.

In [ ]:
# Using both elbow method and silhouette score to find the optimal number of clusters
inertia = []
silhouette_scores = []
K = range(2, 16)
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(nashville_restaurants_clustering)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(nashville_restaurants_clustering, kmeans.labels_))

# Plotting the elbow method and silhouette score  
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(K, inertia, marker='o')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of Clusters')
axes[0].set_ylabel('Inertia')
axes[1].plot(K, silhouette_scores, marker='o')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters')
axes[1].set_ylabel('Silhouette Score')
plt.show()

There is a lot of noise in the data, so pointing out an obvious k-value isn't feasible. However in the elbow method plot, we see a sharp decrease from $k=4$ to $k=5$. With $k=5$, we see a local peak when observing the silhouette scores. Mathematically, $k=2$ gives us the highest silhouette score but it's not realistic when put in a real-world context. Moving forward, we'll use $k=5$.

In [ ]:
# k-means with 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(nashville_restaurants_clustering)
nashville_restaurants_clustering['cluster'] = kmeans.labels_
nashville_restaurants['cluster'] = kmeans.labels_
nashville_restaurants_without_postal = nashville_restaurants.drop(columns=['postal_code'])
nashville_restaurants_without_postal.head(20)

### Visualizing the Clusters

In [ ]:
tiler = cimgt.OSM()
mercator = tiler.crs

fig = plt.figure(figsize=(8, 8))
ax = plt.axes(projection=mercator)
ax.set_extent([-87.0, -86.6, 36.0, 36.3], crs=ccrs.Geodetic())

# Add basemap with roads
ax.add_image(tiler, 12)

# Merge coordinates back in
nashville_restaurants_without_postal = nashville_restaurants_without_postal.merge(
    business_df[['longitude', 'latitude']],
    left_index=True,
    right_index=True
)

# Scatter plot using Cartopy's axis
scatter = ax.scatter(
    nashville_restaurants_without_postal['longitude'],
    nashville_restaurants_without_postal['latitude'],
    c=nashville_restaurants_without_postal['cluster'],
    s=10,
    cmap='hsv',
    alpha=0.7,
    transform=ccrs.Geodetic()
)

ax.set_title('Clusters of Nashville Restaurants', fontsize=15)
legend_labels = [f'Cluster {i}' for i in range(5)]
handles = [plt.Line2D([0], [0], marker='o', color='w', label=label,
                        markerfacecolor=scatter.cmap(i / 5), markersize=10) for i, label in enumerate(legend_labels)]
ax.legend(handles=handles, title='Clusters', loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()


There are some observations that we can make here. The red points (cluster 0) are mostly clustered together around downtown Nashville. Clusters 2, 3, and 4 are clustered along the state roads and US highways. This may reflect restaurants of different price ranges being located in different parts of the city, but we will have to investigate the clusters to see if that claim holds true.

### Investigating the Clusters

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt

def plot_cluster_on_map(df, cluster_col='cluster', cluster_id=0, 
                        lon_col='longitude', lat_col='latitude', 
                        extent=[-87.0, -86.6, 36.0, 36.3],
                        zoom=12, c='darkred'):
    
    # Filter just the cluster
    cluster_df = df[df[cluster_col] == cluster_id]

    # Setting up the map
    tiler = cimgt.OSM()
    mercator = tiler.crs
    fig = plt.figure(figsize=(10, 10))
    ax = plt.axes(projection=mercator)
    ax.set_extent(extent, crs=ccrs.Geodetic())
    ax.add_image(tiler, zoom)

    # Plot points
    ax.scatter(
        cluster_df[lon_col],
        cluster_df[lat_col],
        s=10,
        color=c,
        alpha=0.7,
        transform=ccrs.Geodetic(),
        label=f'Cluster {cluster_id}'
    )

    ax.set_title(f"Restaurants in Cluster {cluster_id} — Nashville", fontsize=15)
    ax.legend()
    plt.show()


In [ ]:
def get_cluster_restaurants(cluster_df, cluster_id, info_df=business_df, cluster_col='cluster'):
    # Get the restaurants in the specified cluster
    extracted_cluster_df = cluster_df[cluster_df[cluster_col] == cluster_id]

    # Get the row numbers of the restaurants in the original DataFrame
    clustered_points_rows = info_df.iloc[extracted_cluster_df.index]

    # Add star ratings, price range, and review count
    extracted_cluster_df['stars'] = clustered_points_rows.loc[:, 'stars']
    extracted_cluster_df['RestaurantsPriceRange2'] = clustered_points_rows.loc[:, 'RestaurantsPriceRange2']
    extracted_cluster_df['review_count'] = clustered_points_rows.loc[:, 'review_count']

    # Drop cluster, longitude, latitude columns
    extracted_cluster_df = extracted_cluster_df.drop(columns=['cluster', 'longitude', 'latitude'])

    return extracted_cluster_df

In [ ]:
# New dataframe with restaurant names and their categories
restaurants_with_categories = pd.read_csv('yelp_business_with_categories.csv')

# Filter for Nashville restaurants
nashville_restaurants_with_categories = restaurants_with_categories[restaurants_with_categories['city'] == 'Nashville']

# Only keep name, address, stars, review count, categories_list, weekday_hours, weekend_hours
nashville_restaurants_with_categories = nashville_restaurants_with_categories[['name', 'address', 'stars', 'review_count', 'RestaurantsPriceRange2', 'categories_list', 'weekday_hours', 'weekend_hours']]

# Function to summarize top 5 categories, average stars, average price range, and average review count for a cluster
def summarize_cluster_data(cluster_df, info_df=nashville_restaurants_with_categories):
    # Get the indices of the restaurants in the cluster_df
    cluster_indices = cluster_df.index.tolist()
    
    # Filter the info_df to get the individual restaurant information in the cluster
    cluster_info_df = info_df.iloc[cluster_indices]
    
    # Parse stringified lists and extract all categories
    all_categories = []
    for row in cluster_info_df['categories_list'].dropna():
        try:
            parsed = ast.literal_eval(row)  # convert string to list
            all_categories.extend([cat.strip() for cat in parsed])
        except (ValueError, SyntaxError):
            continue  # skip any bad rows

    # Get top 5 categories
    top_categories = [cat for cat, _ in Counter(all_categories).most_common(5)]
    
    # Calculate average stars, price range, and review count, average weekday and weekend hours
    avg_stars = cluster_info_df['stars'].mean()
    avg_price_range = cluster_info_df['RestaurantsPriceRange2'].mean()
    med_review_count = cluster_info_df['review_count'].median()
    avg_weekday_hours = cluster_info_df['weekday_hours'].mean()
    avg_weekend_hours = cluster_info_df['weekend_hours'].mean()
    
    
    # Create a summary dictionary
    summary = {
        'Top Categories': top_categories,
        'Average Stars': round(avg_stars, 2),
        'Average Price Range': round(avg_price_range, 2),
        'Median Review Count': round(med_review_count, 1),
        'Average Weekday Hours': round(avg_weekday_hours, 2),
        'Average Weekend Hours': round(avg_weekend_hours, 2)
    }

    return summary

def print_summary(summary: dict):
    # Make this print pretty lol
    print("Top 5 Categories:")
    for category in summary['Top Categories']:
        print(f"-\t{category}")
        
    print()
    print(f"Average Stars:         {summary['Average Stars']}")
    print(f"Average Price Range:   {summary['Average Price Range']}")
    print(f"Median Review Count:   {summary['Median Review Count']}")
    print(f"Average Weekday Hours: {summary['Average Weekday Hours']}")
    print(f"Average Weekend Hours: {summary['Average Weekend Hours']}")

#### Cluster 0


In [ ]:
# Plot cluster 0 restaurants
plot_cluster_on_map(nashville_restaurants_without_postal, cluster_id=0, zoom=10)

In [ ]:
# Get information on cluster 0 restaurants
cluster_0_restaurants = get_cluster_restaurants(nashville_restaurants_without_postal, cluster_id=0)

# Summarize the cluster data
cluster_0_summary = summarize_cluster_data(cluster_0_restaurants)
print("Cluster 0 Summary")
print_summary(cluster_0_summary)

In [ ]:
cluster_0_restaurants_simplified = cluster_0_restaurants[['name', 'address', 'stars', 'review_count', 'RestaurantsPriceRange2']]
cluster_0_restaurants_simplified.head(10)

#### Cluster 1

In [ ]:
# Plot cluster 1 restaurants
plot_cluster_on_map(nashville_restaurants_without_postal, cluster_id=1, zoom=11, c='darkblue')

In [ ]:
# # Get information on cluster 1 restaurants
cluster_1_restaurants = get_cluster_restaurants(nashville_restaurants_without_postal, cluster_id=1)

# Summarize the cluster data
cluster_1_summary = summarize_cluster_data(cluster_1_restaurants)
print("Cluster 1 Summary")
print_summary(cluster_1_summary)

In [ ]:
cluster_1_restaurants_simplified = cluster_1_restaurants[['name', 'address', 'stars', 'review_count', 'RestaurantsPriceRange2']]
cluster_1_restaurants_simplified.head(10)

#### Cluster 2

In [ ]:
# Plot cluster 2 restaurants
plot_cluster_on_map(nashville_restaurants_without_postal, cluster_id=2, zoom=11, c='black')

In [ ]:
# Get information on cluster 2 restaurants
cluster_2_restaurants = get_cluster_restaurants(nashville_restaurants_without_postal, cluster_id=2)

# Summarize the cluster data
cluster_2_summary = summarize_cluster_data(cluster_2_restaurants)
print("Cluster 2 Summary")
print_summary(cluster_2_summary)

In [ ]:
cluster_2_restaurants_simplified = cluster_2_restaurants[['name', 'address', 'stars', 'review_count', 'RestaurantsPriceRange2']]
cluster_2_restaurants_simplified.head(10)

#### Cluster 3

In [ ]:
# Plot cluster 3 restaurants
plot_cluster_on_map(nashville_restaurants_without_postal, cluster_id=3, zoom=11, c='darkgreen')

In [ ]:
# Get information on cluster 3 restaurants
cluster_3_restaurants = get_cluster_restaurants(nashville_restaurants_without_postal, cluster_id=3)

# Summarize the cluster data
cluster_3_summary = summarize_cluster_data(cluster_3_restaurants)
print("Cluster 3 Summary")
print_summary(cluster_3_summary)

In [ ]:
cluster_3_restaurants_simplified = cluster_3_restaurants[['name', 'address', 'stars', 'review_count', 'RestaurantsPriceRange2']]
cluster_3_restaurants_simplified.head(10)

#### Cluster 4

In [ ]:
# Plot cluster 4 restaurants
plot_cluster_on_map(nashville_restaurants_without_postal, cluster_id=4, zoom=11, c='purple')

In [ ]:
# Get information on cluster 4 restaurants
cluster_4_restaurants = get_cluster_restaurants(nashville_restaurants_without_postal, cluster_id=4)

# Summarize the cluster data
cluster_4_summary = summarize_cluster_data(cluster_4_restaurants)
print("Cluster 4 Summary")
print_summary(cluster_4_summary)

In [ ]:
cluster_4_restaurants_simplified = cluster_4_restaurants[['name', 'address', 'stars', 'review_count', 'RestaurantsPriceRange2']]
cluster_4_restaurants_simplified.head(10)